In [27]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()

In [35]:
data = [
    (1, "Alice",25),
    (2, "Bob",30),
    (3, "Charlie",35),
    (4, "David",28),
    (5, "Eve",32),
]
columns = ["Id", "Name", "Age"]
df = spark.createDataFrame(data, columns);

In [36]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"    
        

In [37]:
age_category_udf = udf(categorize_age, StringType())


In [39]:
df_method1 = df.withColumn("Category", age_category_udf(col("Age")))
df_method1.show()

+---+-------+---+--------+
| Id|   Name|Age|Category|
+---+-------+---+--------+
|  1|  Alice| 25|   Adult|
|  2|    Bob| 30|   Adult|
|  3|Charlie| 35|   Adult|
|  4|  David| 28|   Adult|
|  5|    Eve| 32|   Adult|
+---+-------+---+--------+



In [43]:
spark.udf.register("sql_categorize_age" , categorize_age, StringType())
df.createOrReplaceTempView("people")

26/06/13 06:56:38 WARN SimpleFunctionRegistry: The function sql_categorize_age replaced a previously registered function.


In [46]:
sql_df = spark.sql('''
    SELECT 
       Name,
       Age,
        sql_categorize_age(Age) AS Category
      FROM people
      ''')
sql_df.show()
       

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 30|   Adult|
|Charlie| 35|   Adult|
|  David| 28|   Adult|
|    Eve| 32|   Adult|
+-------+---+--------+

